# Differential Expression Analysis

Compare gene expression between conditions (e.g., treatment vs. control).

This notebook:
1. Loads clustered h5ad data
2. Runs differential expression between specified groups
3. Generates volcano plots and MA plots
4. Exports DE results as CSV

In [ ]:
# Parameters (injected by Papermill)
input_h5ad_path: str = "/data/results/experiment/03_clustered.h5ad"
experiment_name: str = "experiment"
groupby: str = "condition"
reference_group: str = "control"
test_group: str = "treatment"
bioaf_api_url: str = "http://localhost:8000"
experiment_id: str = ""

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, frameon=False, figsize=(6, 4))

results_dir = f"/data/results/{experiment_name}"
figures_dir = os.path.join(results_dir, "figures", "de")
os.makedirs(figures_dir, exist_ok=True)

## 1. Load Data & Validate Groups

In [ ]:
adata = sc.read_h5ad(input_h5ad_path)
print(f"Loaded: {adata.n_obs} cells x {adata.n_vars} genes")

if groupby not in adata.obs.columns:
    available = list(adata.obs.columns)
    raise ValueError(
        f"Column '{groupby}' not found in adata.obs. "
        f"Available columns: {available}"
    )

groups = adata.obs[groupby].unique().tolist()
print(f"Groups in '{groupby}': {groups}")
print(f"Reference: {reference_group}, Test: {test_group}")

for g in [reference_group, test_group]:
    if g not in groups:
        raise ValueError(f"Group '{g}' not found. Available: {groups}")

## 2. Differential Expression

Wilcoxon rank-sum test comparing test vs. reference group.

In [ ]:
# Subset to the two groups of interest
mask = adata.obs[groupby].isin([reference_group, test_group])
adata_sub = adata[mask].copy()
print(f"Cells in comparison: {adata_sub.n_obs}")
print(adata_sub.obs[groupby].value_counts())

sc.tl.rank_genes_groups(
    adata_sub,
    groupby=groupby,
    groups=[test_group],
    reference=reference_group,
    method="wilcoxon",
    use_raw=True,
)
print("DE analysis complete")

In [ ]:
# Extract results into a DataFrame
de_df = sc.get.rank_genes_groups_df(adata_sub, group=test_group)
de_df["-log10_pval"] = -np.log10(de_df["pvals"].clip(lower=1e-300))
de_df["-log10_padj"] = -np.log10(de_df["pvals_adj"].clip(lower=1e-300))

n_up = (de_df["logfoldchanges"] > 1).sum()
n_down = (de_df["logfoldchanges"] < -1).sum()
n_sig = (de_df["pvals_adj"] < 0.05).sum()
print(f"Significant genes (padj < 0.05): {n_sig}")
print(f"Upregulated (logFC > 1): {n_up}")
print(f"Downregulated (logFC < -1): {n_down}")
de_df.head(20)

## 3. Volcano Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Classify points
sig_mask = de_df["pvals_adj"] < 0.05
up_mask = sig_mask & (de_df["logfoldchanges"] > 1)
down_mask = sig_mask & (de_df["logfoldchanges"] < -1)
ns_mask = ~(up_mask | down_mask)

ax.scatter(de_df.loc[ns_mask, "logfoldchanges"], de_df.loc[ns_mask, "-log10_padj"],
           s=3, alpha=0.3, c="grey", label="NS")
ax.scatter(de_df.loc[up_mask, "logfoldchanges"], de_df.loc[up_mask, "-log10_padj"],
           s=5, alpha=0.5, c="firebrick", label=f"Up ({up_mask.sum()})")
ax.scatter(de_df.loc[down_mask, "logfoldchanges"], de_df.loc[down_mask, "-log10_padj"],
           s=5, alpha=0.5, c="steelblue", label=f"Down ({down_mask.sum()})")

ax.axhline(-np.log10(0.05), ls="--", color="grey", lw=0.8)
ax.axvline(1, ls="--", color="grey", lw=0.8)
ax.axvline(-1, ls="--", color="grey", lw=0.8)
ax.set_xlabel("Log2 Fold Change")
ax.set_ylabel("-log10(adjusted p-value)")
ax.set_title(f"Volcano Plot: {test_group} vs {reference_group}")
ax.legend(markerscale=3)

# Label top genes
top_genes = de_df.nlargest(10, "-log10_padj")
for _, row in top_genes.iterrows():
    ax.annotate(row["names"], (row["logfoldchanges"], row["-log10_padj"]),
                fontsize=7, alpha=0.8)

fig.savefig(os.path.join(figures_dir, "volcano_plot.png"), bbox_inches="tight")
plt.show()

## 4. MA Plot

In [ ]:
# Compute mean expression for MA plot
if hasattr(adata_sub.raw, "X"):
    raw_X = adata_sub.raw.X.toarray() if hasattr(adata_sub.raw.X, "toarray") else np.array(adata_sub.raw.X)
    mean_expr = pd.Series(raw_X.mean(axis=0), index=adata_sub.raw.var_names)
else:
    X = adata_sub.X.toarray() if hasattr(adata_sub.X, "toarray") else np.array(adata_sub.X)
    mean_expr = pd.Series(X.mean(axis=0), index=adata_sub.var_names)

de_df["mean_expression"] = de_df["names"].map(mean_expr).values

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(de_df["mean_expression"], de_df["logfoldchanges"], s=3, alpha=0.3, c="grey")
sig = de_df[de_df["pvals_adj"] < 0.05]
ax.scatter(sig["mean_expression"], sig["logfoldchanges"], s=5, alpha=0.5, c="firebrick")
ax.axhline(0, ls="-", color="black", lw=0.5)
ax.set_xlabel("Mean Expression")
ax.set_ylabel("Log2 Fold Change")
ax.set_title(f"MA Plot: {test_group} vs {reference_group}")

fig.savefig(os.path.join(figures_dir, "ma_plot.png"), bbox_inches="tight")
plt.show()

## 5. Save Results

In [ ]:
# Save DE table
de_csv = os.path.join(results_dir, f"04_de_{test_group}_vs_{reference_group}.csv")
de_df.to_csv(de_csv, index=False)

# Save h5ad with DE results
output_path = os.path.join(results_dir, "04_de_results.h5ad")
adata_sub.write_h5ad(output_path)

print(f"Saved DE table to: {de_csv}")
print(f"Saved h5ad to: {output_path}")
print(f"Total DE genes (padj < 0.05): {n_sig}")